# Case study bedroom (COCO 000000000632): crop vs. circle

In [ ]:
import sys
sys.path.insert(0, "/workspaces/Object-Centric-Concept-Bottlenecks")

import torch
import open_clip
import splice
from splice.model import SPLICE
from PIL import Image
import torchvision.transforms.functional as TF
import torchvision.transforms.v2 as T
from torchvision.models.detection import maskrcnn_resnet50_fpn
import matplotlib.pyplot as plt
import os

from utils import generate_object_propsals, gather_object_embeddings, draw_circle_on_image
from eval_utils import rank_concepts

In [ ]:
maskrcnn = maskrcnn_resnet50_fpn(pretrained=True)
maskrcnn.eval()
print("MaskRCNN loaded")

concepts = torch.load("/root/.cache/splice/embeddings/open_clip:ViT-B-32_laion_10000_embeddings.pt", map_location="cpu")
image_mean = torch.load("/root/.cache/splice/means/open_clip_ViT-B-32_image.pt", map_location="cpu")
clip_backbone, _, splice_preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="laion2b_s34b_b79k")
clip_backbone.eval()
splice_model = SPLICE(image_mean=image_mean, dictionary=concepts, clip=clip_backbone, device="cpu", l1_penalty=0.25, return_weights=True)
splice_model.eval()
vocab = splice.get_vocabulary("laion", vocabulary_size=10000)
print("SpLiCE loaded")

In [ ]:
import torchvision.transforms as T_v1
from torchvision.models.detection import MaskRCNN_ResNet50_FPN_Weights
from utils import obj_enc_aggregation

IMG_DIR = "/workspaces/Object-Centric-Concept-Bottlenecks/datasets/coco/images/val2017"

def top_concepts(emb, vocab, top_k=5):
    idx = emb.argsort(descending=True)[:top_k]
    return [(vocab[i], emb[i].item()) for i in idx]

P_FNAME = "000000000632.jpg"
p_pil = Image.open(os.path.join(IMG_DIR, P_FNAME)).convert("RGB")
p_img = T_v1.Compose([T_v1.Resize((224, 224)), T_v1.ToTensor()])(p_pil) 

p_boxes, p_scores = generate_object_propsals(p_img.unsqueeze(0), maskrcnn, 0.01, 0.85, 0.5)
p_boxes, p_scores = p_boxes[0], p_scores[0]

pipeline_transform = T.Compose([
    T.Resize(size=224, interpolation=T.InterpolationMode.BICUBIC, max_size=None, antialias=True),
    T.CenterCrop(size=(224, 224)),
    T.Normalize(mean=(0.48145466, 0.4578275, 0.40821073),
                std=(0.26862954, 0.26130258, 0.27577711)),
])

p_kwargs = dict(
    boxes=p_boxes, scores=p_scores, min_score=0.2, num_objects=7, image=p_img,
    device="cpu", splice_model=splice_model, embedding_dim=10000, transform=pipeline_transform,
)
p_crop_embs,   p_used_boxes, p_n = gather_object_embeddings(**p_kwargs, draw_boxes=False, draw_circles=False)
p_circle_embs, _,            _   = gather_object_embeddings(**p_kwargs, draw_boxes=False, draw_circles=True)

full_preprocess = splice.get_preprocess("open_clip:ViT-B-32")
with torch.no_grad():
    p_full_emb = splice_model.encode_image(full_preprocess(p_pil).unsqueeze(0))[0].squeeze(0)

coco_cats = MaskRCNN_ResNet50_FPN_Weights.COCO_V1.meta["categories"]
with torch.no_grad():
    raw = maskrcnn(p_img.unsqueeze(0))[0]
p_labels = []
for box in p_used_boxes:
    k = next(k for k in range(len(raw["boxes"])) if torch.equal(raw["boxes"][k], box))
    p_labels.append(coco_cats[raw["labels"][k]])

print(f"Original size: {p_pil.size}  ->  224x224,  objects used: {p_n}\n")
print(f"{'Obj':<5} {'Class':<15} {'Score':>6}   Box (x1, y1, x2, y2) in 224x224")
print("-" * 60)
for j, (box, lab) in enumerate(zip(p_used_boxes, p_labels)):
    print(f"{j+1:<5} {lab:<15} {p_scores[j].item():>6.3f}   {[round(v, 1) for v in box.tolist()]}")

In [ ]:
def denorm(t):
    mean = torch.tensor((0.48145466, 0.4578275, 0.40821073)).view(3, 1, 1)
    std = torch.tensor((0.26862954, 0.26130258, 0.27577711)).view(3, 1, 1)
    return (t * std + mean).clamp(0, 1).permute(1, 2, 0)

fig, axes = plt.subplots(2, p_n, figsize=(2.6 * p_n, 6))
for j, (box, lab) in enumerate(zip(p_used_boxes, p_labels)):
    x1, y1, x2, y2 = box.long()
    crop_in = pipeline_transform(p_img[:, y1:y2, x1:x2].unsqueeze(0)).squeeze(0)                           
    circle_in = pipeline_transform(TF.to_tensor(draw_circle_on_image(p_img, box)).unsqueeze(0)).squeeze(0)  
    axes[0, j].imshow(denorm(crop_in))
    axes[0, j].set_title(f"Obj {j+1}: {lab}\n{p_scores[j]:.2f}", fontsize=10)
    axes[1, j].imshow(denorm(circle_in))
    for ax in axes[:, j]:
        ax.set_xticks([]); ax.set_yticks([])
axes[0, 0].set_ylabel("Crop", fontsize=12)
axes[1, 0].set_ylabel("Circle", fontsize=12)
plt.suptitle("Image 3: CLIP inputs after pipeline preprocessing (224×224, normalization undone)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
def fmt(c):
    return f"{c[0]} ({c[1]:.3f})"

print("Full image")
print("-" * 30)
for c in top_concepts(p_full_emb, vocab, top_k=5):
    print(f"  {fmt(c)}")

for j, lab in enumerate(p_labels):
    crop_c = top_concepts(p_crop_embs[j], vocab, top_k=5)
    circ_c = top_concepts(p_circle_embs[j], vocab, top_k=5)
    print(f"\nObject {j+1}: {lab} ({p_scores[j]:.2f})")
    print(f"  {'Crop':<28} {'Circle':<28}")
    print("  " + "-" * 56)
    for a, b in zip(crop_c, circ_c):
        print(f"  {fmt(a):<28} {fmt(b):<28}")

In [ ]:
p_crop_b, p_circle_b, p_full_b = p_crop_embs.unsqueeze(0), p_circle_embs.unsqueeze(0), p_full_emb.unsqueeze(0)

aggs = {
    "Crop + Sum":     obj_enc_aggregation("sum",   p_crop_b,   p_full_b).squeeze(0),
    "Circle + Sum":   obj_enc_aggregation("sum",   p_circle_b, p_full_b).squeeze(0),
    "Crop + Count":   obj_enc_aggregation("count", p_crop_b,   p_full_b).squeeze(0),
    "Circle + Count": obj_enc_aggregation("count", p_circle_b, p_full_b).squeeze(0),
}
tops = {name: top_concepts(v, vocab, top_k=10) for name, v in aggs.items() if "Sum" in name}
for m in ("Crop", "Circle"):
    s, cnt = aggs[f"{m} + Sum"].numpy(), aggs[f"{m} + Count"].numpy()
    tops[f"{m} + Count"] = [(vocab[i], cnt[i]) for i in rank_concepts(s, cnt, "count")[:10]]

print(f"Full image + {p_n} objects  (Count: number of sources, max = {p_n + 1})\n")
print("".join(f"{name:<26}" for name in tops))
print("-" * 104)
for i in range(10):
    row = ""
    for name, t in tops.items():
        w, v = t[i]
        row += f"{w} ({v:.3f})".ljust(26) if "Sum" in name else f"{w} ({int(v)})".ljust(26)
    print(row)

In [ ]:
import torch.nn.functional as F

print(f"Cosine similarity between object concept vector and full-image concept vector\n")
print(f"{'Obj':<5} {'Class':<15} {'Crop':>8} {'Circle':>8}")
print("-" * 40)
cos_crop = F.cosine_similarity(p_crop_embs, p_full_emb.unsqueeze(0), dim=1)
cos_circle = F.cosine_similarity(p_circle_embs, p_full_emb.unsqueeze(0), dim=1)
for j, lab in enumerate(p_labels):
    print(f"{j+1:<5} {lab:<15} {cos_crop[j].item():>8.3f} {cos_circle[j].item():>8.3f}")
print("-" * 40)
print(f"{'Mean':<21} {cos_crop.mean().item():>8.3f} {cos_circle.mean().item():>8.3f}")

In [ ]:
os.makedirs("figures", exist_ok=True)

fig, axes = plt.subplots(2, p_n, figsize=(14, 4.6), gridspec_kw=dict(wspace=0.04, hspace=0.04))
for j, (box, lab, score) in enumerate(zip(p_used_boxes, p_labels, p_scores)):
    x1, y1, x2, y2 = box.long()
    crop_in = pipeline_transform(p_img[:, y1:y2, x1:x2].unsqueeze(0)).squeeze(0)
    circle_in = pipeline_transform(TF.to_tensor(draw_circle_on_image(p_img, box)).unsqueeze(0)).squeeze(0)
    axes[0, j].imshow(denorm(crop_in))
    axes[1, j].imshow(denorm(circle_in))
    axes[0, j].set_title(f"{j+1} {lab} ({score.item():.2f})", fontsize=11)
    for ax in axes[:, j]:
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
axes[0, 0].set_ylabel("Crop", fontsize=12)
axes[1, 0].set_ylabel("Marked", fontsize=12)

fig.savefig("figures/case_study_bedroom.pdf", bbox_inches="tight", dpi=300)
fig.savefig("figures/case_study_bedroom.png", bbox_inches="tight", dpi=300)
plt.show()

for f in ["figures/case_study_bedroom.pdf", "figures/case_study_bedroom.png"]:
    print(f"{os.path.abspath(f)}: {os.path.getsize(f) / 1024:.1f} KB")